In [0]:
WITH data_prep AS (
    select instrument as stock_name,
        substr(ticker,1,instr(ticker,'.')-1) as ticker,    
        case type
            when 'Stock purchase' then 'BUY'
            when 'Stock sell' then 'SELL'
            else 'OTHER'
        end as trans_type, 
        category, 
        to_date(time,'YYYY/MM/DD') as trans_date, 
        cast(replace(amount,',','.') as double) as trans_amount, 
        --comment,
        cast(substr(comment,instr(comment,'@')+2) as double) as stock_price,
        round(cast(replace(amount,',','.') as double)/cast(substr(comment,instr(comment,'@')+2) as double),2)*-1 as stock_volume
    from stock_lakehouse.source_layer.us_stock_cash_raw
    where instrument = "Oracle"
    and type like "Stock%"
    and time >= "2026-08-03T15:35:07.000+00:00" 
    order by time desc
)
select stock_name, sum(stock_volume)
from data_prep
group by stock_name;